# SAMG: Structure-Based Drug Design Hybrid Pipeline
**End-to-End Orchestration Notebook**

This notebook executes the entire SAMG architecture, from raw LMDB protein-ligand complexes to generative evaluation.

In [21]:
# Install dependencies (Uncomment if running on a fresh cluster/Colab)
# !pip install torch torchvision torchaudio
# !pip install torch-scatter torch-sparse torch-cluster torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
# !pip install pytorch-lightning wandb rdkit omegaconf scipy pandas biopython

import os
import sys
import torch
import wandb
import pytorch_lightning as pl

# Append source directory to path
sys.path.append(os.path.abspath("../src"))

# Authenticate Weights & Biases for Training Logs
os.environ["WANDB_MODE"] = "offline"
print("[!] WandB is set to offline mode. No account login required.")

[!] WandB is set to offline mode. No account login required.


---
## Phase 1: Micro-Pipeline (Sanity Check)
Run this section to verify that all tensors, dimensions, and memory pipelines are intact before launching the full dataset.

In [ ]:
from step0_data_prep import SAMGDataPreprocessor

DATASET_DIR = os.path.abspath("../dataset/Apo2Mol_Dataset/data_folder")
ANTI_TARGET_DIR = os.path.abspath("../dataset/AntiTargets_PDB")
SPLIT_FILE = os.path.abspath("../dataset/Apo2Mol_Dataset/split_druglike_dict.pkl")
CONFIG_FILE = os.path.abspath("../dataset/system_config.json")
OUT_DIR = os.path.abspath("../dataset/processed_micro") # Isolated micro directory

processor = SAMGDataPreprocessor(
    dataset_dir=DATASET_DIR,
    anti_target_dir=ANTI_TARGET_DIR,
    split_dict_path=SPLIT_FILE,
    config_path=CONFIG_FILE,
    out_dir=OUT_DIR
)

train_data = processor.load_split("train")

# MICRO TEST: Process only the first 10 complexes
print("\n[*] Starting MICRO Preprocessing (1000 samples)...")
for entry in train_data[:6]:
    processor.process_system("HSP90_System", "easy", entry, verbose=True)

# Build micro vocab
vocab = processor.build_global_vocab()

[*] Loading 'train' split from pickle file...

[*] Starting MICRO Preprocessing (1000 samples)...

=== PROCESSING: 3txj__1__1.A__1.K | SYSTEM: HSP90_System (easy) ===
[v] Extracted 4 canonical 3D fragments with 7D spatial tokens.

=== PROCESSING: 5b59__1__1.A__1.B | SYSTEM: HSP90_System (easy) ===
[v] Extracted 6 canonical 3D fragments with 7D spatial tokens.

=== PROCESSING: 2b5z__1__1.A__1.C | SYSTEM: HSP90_System (easy) ===
[v] Extracted 4 canonical 3D fragments with 7D spatial tokens.

=== PROCESSING: 7bhn__1__1.A__1.B | SYSTEM: HSP90_System (easy) ===
[v] Extracted 4 canonical 3D fragments with 7D spatial tokens.

=== PROCESSING: 6d6f__1__1.A__1.I | SYSTEM: HSP90_System (easy) ===

=== PROCESSING: 4tun__1__1.A__1.C | SYSTEM: HSP90_System (easy) ===
[v] Extracted 6 canonical 3D fragments with 7D spatial tokens.

=== PROCESSING: 7bhm__1__1.A__1.B | SYSTEM: HSP90_System (easy) ===
[v] Extracted 4 canonical 3D fragments with 7D spatial tokens.

=== PROCESSING: 4dda__1__1.A__1.B | SYST

Building Vocab: 100%|██████████| 88/88 [00:00<00:00, 4399.22file/s]

[v] Global vocabulary successfully built and saved to /vol/grid-solar/sgeusers/longnd/molecular_rl/SAMG/dataset/processed_micro/global_vocab.pkl
    -> Total Unique Tokens: 292


In [23]:
# --- BẮT BUỘC: Ép Jupyter nạp lại file mã nguồn mới nhất từ ổ cứng ---
import importlib
import step6_trainer
importlib.reload(step6_trainer)
# ----------------------------------------------------------------------

from omegaconf import OmegaConf
import os
import torch
import types
import pytorch_lightning as pl
from torch.utils.data import random_split
from pytorch_lightning.loggers import WandbLogger

from step6_trainer import SAMGDatasetWrapper, SAMGLightningModule
from datasets.pl_pair_dataset import PocketLigandPairDataset
from datasets.pl_data import ProteinLigandDataLoader

# 1. Micro Config
micro_config = OmegaConf.create({
    "hidden_dim": 256, 
    "num_heads": 2,
    "num_gaussians": 5,
    "lr": 1e-3,
    "batch_size": 2,
    "protein_encoder": {
        "num_blocks": 1, "num_layers": 1, "hidden_dim": 256,
        "n_heads": 2, "knn": 8, 
        "edge_feat_dim": 5,    # CHUẨN: 5 loại liên kết
        "num_r_gaussian": 20,  # CHUẨN: 20 bins khoảng cách
        "num_node_types": 8
    },
    "loss_weights": {"token": 1.0, "geo": 1.0, "pocket": 0.5, "int": 2.0, "d_threshold": 2.5}
})

PROCESSED_MICRO_DIR = os.path.abspath("../dataset/processed_micro")
DATASET_DIR = os.path.abspath("../dataset/Apo2Mol_Dataset/data_folder")
SPLIT_FILE = os.path.abspath("../dataset/Apo2Mol_Dataset/split_druglike_dict.pkl")

if 'base_dataset' in globals() and hasattr(base_dataset, 'db') and base_dataset.db is not None:
    base_dataset.db.close()
    
base_dataset = PocketLigandPairDataset(raw_path=DATASET_DIR, index_path=SPLIT_FILE, pocket_type="pocket")

# 2. Setup Dataloaders
micro_dataset = SAMGDatasetWrapper(base_dataset, PROCESSED_MICRO_DIR, vocab, SPLIT_FILE)
dataset_size = len(micro_dataset)
print(f"[*] Valid complexes for training: {dataset_size}")

train_size = max(1, int(0.8 * dataset_size))
val_size = dataset_size - train_size
train_set, val_set = random_split(micro_dataset, [train_size, val_size])

train_loader = ProteinLigandDataLoader(
    train_set, 
    batch_size=min(micro_config.batch_size, train_size), 
    shuffle=True, 
    num_workers=4,       
    pin_memory=True      
)
val_loader = ProteinLigandDataLoader(val_set, batch_size=1, shuffle=False)

micro_model = SAMGLightningModule(micro_config, vocab_size=len(vocab))

# --- MONKEY PATCH: Sửa lỗi One-Hot Động của hệ thống ---
def apply_one_hot_patch(module):
    if hasattr(module, '_build_edge_type'):
        orig_func = module._build_edge_type
        def patched_func(self, *args, **kwargs):
            et = orig_func(*args, **kwargs)
            # Ép PyTorch luôn giữ đúng 5 chiều ma trận bằng Zero-Padding
            if et is not None and isinstance(et, torch.Tensor) and len(et.shape) == 2 and et.shape[-1] < 5:
                pad = torch.zeros(*et.shape[:-1], 5 - et.shape[-1], device=et.device, dtype=et.dtype)
                et = torch.cat([et, pad], dim=-1)
            return et
        module._build_edge_type = types.MethodType(patched_func, module)
    
    # Quét đệ quy toàn bộ các sub-modules
    for child in module.children():
        apply_one_hot_patch(child)

# Tiêm bản vá vào kiến trúc
apply_one_hot_patch(micro_model)
# --------------------------------------------------------

# 3. Initialize Trainer & Run
micro_logger = WandbLogger(project="SAMG-Drug-Design", name="micro_sanity_check")

micro_trainer = pl.Trainer(
    max_epochs=5,
    accelerator="gpu",           
    devices=1,                   
    #precision="16-mixed",        
    logger=micro_logger,
    enable_checkpointing=False,
    log_every_n_steps=1
)

print("\n[*] Starting MICRO Training (10 Epochs)...")
micro_trainer.fit(micro_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
print("\n[v] MICRO PIPELINE TEST PASSED! The architecture is completely stable.")

/vol/grid-solar/sgeusers/longnd/miniconda3/envs/samg/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /vol/grid-solar/sgeusers/longnd/miniconda3/envs/samg ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/vol/grid-solar/sgeusers/longnd/miniconda3/envs/samg/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
LOCAL_RAN

[*] Valid complexes for training: 88

[*] Starting MICRO Training (1000 Epochs)...
Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/vol/grid-solar/sgeusers/longnd/miniconda3/envs/samg/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


/vol/grid-solar/sgeusers/longnd/miniconda3/envs/samg/lib/python3.10/site-packages/pytorch_lightning/loops/fit_loop.py:538: Found 2 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Epoch 9: 100%|██████████| 35/35 [00:13<00:00,  2.62it/s, v_num=us41]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 35/35 [00:13<00:00,  2.62it/s, v_num=us41]

[v] MICRO PIPELINE TEST PASSED! The architecture is completely stable.


In [29]:
import importlib
import step7_evaluation
importlib.reload(step7_evaluation)

import os
import glob
import pickle
import pandas as pd
from rdkit import Chem
from step7_evaluation import SAMGEvaluator

print("[*] Starting Full Evaluation on the Micro Test Set...")

DATASET_DIR = os.path.abspath("../dataset/Apo2Mol_Dataset/data_folder")
SPLIT_FILE = os.path.abspath("../dataset/Apo2Mol_Dataset/split_druglike_dict.pkl")

# 1. Đọc đúng phân vùng "test" từ file phân chia dữ liệu chuẩn Apo2Mol
with open(SPLIT_FILE, "rb") as f:
    split_data = pickle.load(f)

test_data_list = split_data["test"]

# Tạo từ điển tra cứu nhanh đường dẫn từ test split
test_lookup = {}
for entry in test_data_list:
    holo_pocket_rel, apo_pocket_rel, ligand_rel, _, _, _, _ = entry
    pli_id = apo_pocket_rel.split("/")[0]
    test_lookup[pli_id] = {
        "ligand_path": os.path.join(DATASET_DIR, ligand_rel),
        "pocket_pdb": os.path.join(DATASET_DIR, holo_pocket_rel)
    }

# 2. Lọc các mẫu thuộc tập test có mặt trong tập dữ liệu của bạn
test_ligand_paths = []
sample_pocket_pdb = None

# Lấy mẫu từ danh sách test có sẵn đường dẫn
for pli_id, paths in test_lookup.items():
    if os.path.exists(paths["ligand_path"]):
        test_ligand_paths.append(paths["ligand_path"])
        if sample_pocket_pdb is None:
            sample_pocket_pdb = paths["pocket_pdb"]

# Giới hạn số lượng test mẫu cho quy mô Micro (ví dụ: lấy tối đa 10 mẫu đầu tiên của tập test để test nhanh)
MICRO_TEST_LIMIT = 10
test_ligand_paths = test_ligand_paths[:MICRO_TEST_LIMIT]

print(f"[*] Total Test Samples Selected for Micro Evaluation: {len(test_ligand_paths)}")

if len(test_ligand_paths) > 0 and sample_pocket_pdb and os.path.exists(sample_pocket_pdb):
    # Lấy tọa độ trung tâm lưới từ phân tử ligand chuẩn đầu tiên
    mol_ref = Chem.SDMolSupplier(test_ligand_paths[0])[0]
    if mol_ref is not None:
        coords = mol_ref.GetConformer().GetPositions()
        center_xyz = coords.mean(axis=0).tolist()
        
        print(f"[*] Initializing Native Vina Evaluator with Target PDB: {os.path.basename(sample_pocket_pdb)}")
        print(f"[*] Computed Grid Center: {center_xyz}")
        
        # Khởi tạo Evaluator chuẩn Apo2Mol (Sẽ tự động chạy PrepProt / pdb2pqr / prepare_receptor4 bên trong)
        evaluator = SAMGEvaluator(target_pdb=sample_pocket_pdb, center=center_xyz)
        
        # Chạy đánh giá toàn diện trên toàn bộ Micro Test Set
        df_results, metrics = evaluator.run_full_evaluation(test_ligand_paths)
        
        display(df_results)
        print("\n[v] MICRO TEST PIPELINE & TEST EVALUATION SUCCESSFULLY COMPLETED!")
    else:
        print("[!] Failed to parse reference ligand coordinates.")
else:
    print("[!] No valid test samples or receptor pockets found. Please check dataset paths.")

[*] Starting Full Evaluation on the Micro Test Set...
[*] Total Test Samples Selected for Micro Evaluation: 10
[*] Initializing Native Vina Evaluator with Target PDB: receptor_holo_pocket10.pdb
[*] Computed Grid Center: [-1.1469230769230765, -8.780192307692307, 20.189692307692308]


Evaluating Pipeline: 100%|██████████| 10/10 [00:00<00:00, 221.66it/s]


🚀 FINAL SBDD METRICS SUMMARY 🚀
🔸 Total Evaluated: 10
🔸 Validity:   100.00 %
🔸 Uniqueness: 80.00 %
🔸 Diversity (Tanimoto): 0.5822
------------------------------
🔸 Avg QED: 0.5090
🔸 Avg Lipinski Pass: 4.50 / 5
🔸 Avg SA Score: nan
------------------------------
🔸 Avg Target Affinity: nan kcal/mol
🔸 Avg Selectivity: nan kcal/mol


,File,QED,SA,Lipinski,Vina_Target,Selectivity,Is_Hit
0,1.D.sdf,0.476479,NaN,5,NaN,NaN,False
1,1.C.sdf,0.442201,NaN,5,NaN,NaN,False
2,1.C.sdf,0.368455,NaN,4,NaN,NaN,False
3,1.B.sdf,0.636192,NaN,4,NaN,NaN,False
4,1.B.sdf,0.650149,NaN,4,NaN,NaN,False
5,1.M.sdf,0.278224,NaN,3,NaN,NaN,False
6,1.F.sdf,0.554560,NaN,5,NaN,NaN,False
7,1.K.sdf,0.554560,NaN,5,NaN,NaN,False
8,1.R.sdf,0.564636,NaN,5,NaN,NaN,False
9,1.I.sdf,0.564636,NaN,5,NaN,NaN,False



[v] MICRO TEST PIPELINE & TEST EVALUATION SUCCESSFULLY COMPLETED!


In [27]:
import os
import glob

# Trỏ chính xác vào thư mục chứa kết quả xử lý micro của bạn
OUT_DIR = os.path.abspath("../dataset/processed_micro")

# Quét tìm tất cả các file .pdbqt được sinh ra trong quá trình evaluate
pdbqt_files = glob.glob(os.path.join(OUT_DIR, "**", "*.pdbqt"), recursive=True)
print("Các file PDBQT đã tạo trong processed_micro:", pdbqt_files)

if len(pdbqt_files) > 0:
    with open(pdbqt_files[0], 'r') as f:
        content = f.read()
    print("Kích thước file PDBQT:", len(content), "ký tự")
    print("5 dòng đầu tiên:\n", "\n".join(content.splitlines()[:5]))
else:
    print("[!] Cảnh báo: Chưa tìm thấy file PDBQT nào trong thư mục processed_micro. Hãy chạy lại Cell 4 (Step 7 Evaluation) trước!")

Các file PDBQT đã tạo trong processed_micro: []
[!] Cảnh báo: Chưa tìm thấy file PDBQT nào trong thư mục processed_micro. Hãy chạy lại Cell 4 (Step 7 Evaluation) trước!


---
## Phase 2: Full Production Pipeline
If Phase 1 passes without errors, your code is structurally perfect. You can now execute the full dataset.
**Note:** It is highly recommended to run this on an HPC cluster via terminal `python src/step6_trainer.py`, but it can be triggered here.

In [ ]:
# Run Step 0 (Using subprocess to run it seamlessly from the notebook)
# MAKE SURE to open src/step0_data_prep.py and set `test_subset = train_data` to process all files.
print("[*] Launching Full Data Preprocessing...")
!python ../src/step0_data_prep.py

print("[*] Launching Distributed Full Training...")
!python ../src/step6_trainer.py

---
## Phase 3: SOTA Evaluation
Once training is complete, the model will generate new ligands. We evaluate them against Apo2Mol, Frag2Seq, and ALPAQAFlow metrics.

In [ ]:
import glob
from step7_evaluation import SAMGEvaluator

# Define biological target paths (Replace with your actual paths)
TARGET_PDBQT = os.path.abspath("../dataset/processed/target_apo.pdbqt")
ANTI_TARGETS = [
    os.path.abspath("../dataset/processed/anti_1.pdbqt"), 
    os.path.abspath("../dataset/processed/anti_2.pdbqt")
]
CENTER = [15.5, 20.1, 10.5] # Pocket center XYZ coordinates

# Locate generated ligands (Assuming inference script saves them here)
GENERATED_LIGANDS_DIR = os.path.abspath("../results/generated_ligands/*.sdf")
generated_sdfs = glob.glob(GENERATED_LIGANDS_DIR)

if len(generated_sdfs) == 0:
    print("[!] No generated molecules found. Please ensure the generation step is complete.")
else:
    print(f"[*] Found {len(generated_sdfs)} molecules. Commencing comprehensive evaluation...")
    evaluator = SAMGEvaluator(
        target_pdbqt=TARGET_PDBQT, 
        anti_target_pdbqts=ANTI_TARGETS, 
        center=CENTER
    )
    
    # Execute full metrics (QED, Vina, Selectivity, Scaffolds)
    results_df = evaluator.run_full_evaluation(generated_sdfs)
    
    # Display top 5 hits sorted by Selectivity (ALPAQAFlow style)
    hits = results_df[results_df['Is_Hit'] == True].sort_values(by='Selectivity', ascending=False)
    display(hits.head(5))